# 📒 Bài 4 — Chống overfit, GPU & Transfer learning

**Tuần 13**

> Bài này cố ý dùng tập train **nhỏ (1.000 ảnh)** để overfitting lộ rõ — rồi chữa từng bước và đo hiệu quả thật.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

MAU = ["#4269d0", "#efb118", "#3ca951", "#ff725c", "#a463f2"]

DATA = Path("../../../data")
if not (DATA / "hinh_khoi.npz").exists():
    DATA = Path("data")

d = np.load(DATA / "hinh_khoi.npz")
TEN_LOP = list(d["ten_lop"])

# CỐ Ý dùng tập train NHỎ (1000 ảnh) để overfitting lộ rõ
X_train = torch.tensor(d["X_train"][:1000]).unsqueeze(1)
y_train = torch.tensor(d["y_train"][:1000])
X_test = torch.tensor(d["X_test"]).unsqueeze(1)
y_test = torch.tensor(d["y_test"])

print(f"train {len(X_train)} ảnh (cố ý ít)   test {len(X_test)} ảnh")


def tao_batch(X, y, bs, seed=0):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(X), generator=g)
    return [(X[idx[i:i+bs]], y[idx[i:i+bs]]) for i in range(0, len(X), bs)]


@torch.no_grad()
def danh_gia(model, X, y, lf):
    model.eval()
    lg = model(X)
    return {"loss": lf(lg, y).item(), "acc": (lg.argmax(1) == y).float().mean().item()}

## 1. Tạo ra overfitting để nhìn thấy nó

In [ ]:
def tao_cnn(dropout=0.0, batchnorm=False):
    lop = [nn.Conv2d(1, 16, 3, padding=1)]
    if batchnorm:
        lop.append(nn.BatchNorm2d(16))
    lop += [nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1)]
    if batchnorm:
        lop.append(nn.BatchNorm2d(32))
    lop += [nn.ReLU(), nn.MaxPool2d(2), nn.Flatten()]
    if dropout:
        lop.append(nn.Dropout(dropout))
    lop += [nn.Linear(32 * 7 * 7, 64), nn.ReLU()]
    if dropout:
        lop.append(nn.Dropout(dropout))
    lop.append(nn.Linear(64, 4))
    return nn.Sequential(*lop)


def train(model, so_epoch=30, lr=1e-3, weight_decay=0.0, augment=None, seed=0):
    lf = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    ls = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for e in range(so_epoch):
        model.train()
        for xb, yb in tao_batch(X_train, y_train, 32, seed=seed + e):
            if augment is not None:
                xb = augment(xb, seed=seed + e)
            loss = lf(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()

        tr = danh_gia(model, X_train, y_train, lf)
        va = danh_gia(model, X_test, y_test, lf)
        ls["train_loss"].append(tr["loss"]); ls["train_acc"].append(tr["acc"])
        ls["val_loss"].append(va["loss"]);   ls["val_acc"].append(va["acc"])
    return ls


torch.manual_seed(0)
ls_tho = train(tao_cnn())

print(f"  train acc: {ls_tho['train_acc'][-1]:.4f}")
print(f"  val   acc: {ls_tho['val_acc'][-1]:.4f}")
print(f"  khoảng cách: {ls_tho['train_acc'][-1] - ls_tho['val_acc'][-1]:+.4f}   ← OVERFIT")

In [ ]:
def ve_duong_cong(cac_ls, tieu_de="Đường cong học"):
    fig, (a1, a2) = plt.subplots(1, 2)
    for i, (ten, ls) in enumerate(cac_ls.items()):
        mau = MAU[i % len(MAU)]
        a1.plot(ls["train_loss"], lw=2, color=mau, ls="--", alpha=.6)
        a1.plot(ls["val_loss"], lw=2, color=mau, label=ten)
        a2.plot(ls["val_acc"], lw=2, color=mau, label=ten)
    a1.set_xlabel("epoch"); a1.set_ylabel("loss")
    a1.set_title("Loss (nét đứt = train, nét liền = val)")
    a2.set_xlabel("epoch"); a2.set_ylabel("val accuracy"); a2.set_title("Độ chính xác")
    for a in (a1, a2):
        a.legend(frameon=False)
    plt.suptitle(tieu_de); plt.tight_layout(); plt.show()


ve_duong_cong({"CNN thô": ls_tho}, "Overfitting kinh điển")

**Đọc biểu đồ:** train loss (nét đứt) lao xuống gần 0, val loss (nét liền) giảm một lúc rồi **tăng trở lại**. Đó là chữ ký của overfitting.

Model đã **học thuộc** 1.000 ảnh train, kể cả nhiễu ngẫu nhiên trong chúng.

## 2. Dropout

In [ ]:
torch.manual_seed(0)
ls_drop = train(tao_cnn(dropout=0.3))

ve_duong_cong({"CNN thô": ls_tho, "+ Dropout 0.3": ls_drop}, "Dropout")
print(f"  Không dropout : val acc {ls_tho['val_acc'][-1]:.4f}")
print(f"  Dropout 0.3   : val acc {ls_drop['val_acc'][-1]:.4f}")

**Dropout tắt ngẫu nhiên x% nơ-ron ở mỗi bước train.** Model không thể dựa vào một vài nơ-ron cụ thể → buộc phải học biểu diễn dư thừa, bền vững hơn.

```
Khi train:            Khi eval:
  ○ ● ○ ● ●             ● ● ● ● ●
  ● ○ ● ○ ●             ● ● ● ● ●
  (tắt ngẫu nhiên)      (dùng toàn bộ)
```

> ⚠️ **Đây là lý do `model.train()` / `model.eval()` bắt buộc.** Quên `eval()` khi đánh giá → dropout vẫn bật → kết quả sai và dao động lung tung giữa các lần chạy. Một trong những bug khó tìm nhất với người mới.

`p = 0.2–0.5` là khoảng thông dụng. Đặt sau lớp Linear lớn, không đặt sau lớp cuối.

## 3. ⭐ Data augmentation — "dữ liệu miễn phí"

In [ ]:
def augment(xb, seed=0):
    """Lật và dịch chuyển ngẫu nhiên — chỉ dùng torch, không cần torchvision."""
    g = torch.Generator().manual_seed(seed)
    out = xb.clone()

    lat = torch.rand(len(xb), generator=g) < 0.5
    out[lat] = torch.flip(out[lat], dims=[3])              # lật ngang

    dich = torch.randint(-3, 4, (2,), generator=g)
    return torch.roll(out, shifts=(int(dich[0]), int(dich[1])), dims=(2, 3))


goc = X_train[:6]
bien_the = augment(goc, seed=1)

fig, axes = plt.subplots(2, 6, figsize=(11, 3.8))
for i in range(6):
    axes[0, i].imshow(goc[i, 0], cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(bien_the[i, 0], cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_ylabel("gốc"); axes[1, 0].set_ylabel("augment")
plt.suptitle("Trên: ảnh gốc — Dưới: sau khi lật + dịch chuyển", y=1.04)
plt.tight_layout(); plt.show()

In [ ]:
torch.manual_seed(0)
ls_aug = train(tao_cnn(), augment=augment)

ve_duong_cong({"CNN thô": ls_tho, "+ Augmentation": ls_aug}, "Data augmentation")
print(f"  Không augment : val acc {ls_tho['val_acc'][-1]:.4f}")
print(f"  Có augment    : val acc {ls_aug['val_acc'][-1]:.4f}")

> 🔑 **Augmentation thường là kỹ thuật chống overfit hiệu quả nhất với ảnh** — vì nó tấn công đúng gốc rễ: model overfit do **thiếu dữ liệu**, và augmentation tạo ra dữ liệu mới gần như miễn phí.

**Quy tắc chọn phép biến đổi: phép biến đổi phải KHÔNG làm đổi nhãn.**

| Bài toán | Lật ngang | Xoay 180° |
|---|---|---|
| Nhận diện hình khối | ✅ | ✅ |
| Nhận diện mèo/chó | ✅ | ❌ (mèo lộn ngược hiếm) |
| **Nhận diện chữ số** | ❌ (6 ↔ ... ) | ❌ (6 → 9) |
| Ảnh y tế | ⚠️ tuỳ (tim nằm bên trái!) | ❌ |

> ⚠️ **Chỉ augment tập TRAIN, không bao giờ augment tập test.** Test phải phản ánh dữ liệu thật.

Trong thực tế bạn dùng `torchvision.transforms` — nhưng nguyên lý đúng như đoạn code trên.

## 4. Weight decay & BatchNorm

In [ ]:
torch.manual_seed(0)
ls_wd = train(tao_cnn(), weight_decay=1e-3)

torch.manual_seed(0)
ls_bn = train(tao_cnn(batchnorm=True))

ve_duong_cong({"CNN thô": ls_tho, "+ Weight decay": ls_wd, "+ BatchNorm": ls_bn},
              "Weight decay & BatchNorm")

**Weight decay** phạt trọng số lớn — chính là **L2 regularization** ở Phase 04, chỉ khác tên. Trong PyTorch nó là một tham số của optimizer.

**BatchNorm** chuẩn hoá đầu ra mỗi lớp về mean 0, std 1 trong từng batch. Tác dụng chính là **train nhanh và ổn định hơn** (cho phép dùng lr lớn hơn); chống overfit chỉ là tác dụng phụ nhẹ.

> ⚠️ BatchNorm cũng hành xử khác giữa `train()` và `eval()` — khi train nó dùng thống kê của batch hiện tại, khi eval dùng trung bình động đã tích luỹ.

## 5. ⭐ Kết hợp tất cả

In [ ]:
torch.manual_seed(0)
ls_full = train(tao_cnn(dropout=0.3, batchnorm=True), weight_decay=1e-4, augment=augment)

ve_duong_cong({"CNN thô": ls_tho, "Đầy đủ": ls_full}, "Thô vs Đầy đủ chống overfit")

print(f"{'':<22} {'train':>8} {'val':>8} {'khoảng cách':>13}")
print("-" * 54)
for ten, ls in [("CNN thô", ls_tho), ("+ Dropout", ls_drop), ("+ Augment", ls_aug),
                ("+ Weight decay", ls_wd), ("+ BatchNorm", ls_bn), ("Đầy đủ", ls_full)]:
    print(f"{ten:<22} {ls['train_acc'][-1]:>8.4f} {ls['val_acc'][-1]:>8.4f} "
          f"{ls['train_acc'][-1] - ls['val_acc'][-1]:>+13.4f}")

> 📌 Cột **khoảng cách** là thứ đáng nhìn nhất: nó nhỏ đi nghĩa là model tổng quát hoá tốt hơn. Nhưng mục tiêu cuối vẫn là **val acc cao nhất**, không phải khoảng cách nhỏ nhất — một model dở đều cả hai bên cũng có khoảng cách nhỏ.

## 6. Early stopping — rẻ và hiệu quả nhất

In [ ]:
# Train LÂU hơn để val loss lộ rõ điểm quay đầu
torch.manual_seed(0)
ls_dai = train(tao_cnn(), so_epoch=60)

epoch_tot_nhat = int(np.argmin(ls_dai["val_loss"]))

plt.plot(ls_dai["val_loss"], lw=2, color=MAU[0], label="val loss")
plt.plot(ls_dai["train_loss"], lw=2, color=MAU[0], ls="--", alpha=.5, label="train loss")
plt.axvline(epoch_tot_nhat, color=MAU[3], ls="--", lw=2,
            label=f"dừng ở đây (epoch {epoch_tot_nhat + 1})")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(frameon=False)
plt.title("Early stopping — dừng tại val loss thấp nhất")
plt.show()

cuoi = len(ls_dai["val_acc"])
print(f"  Epoch tốt nhất {epoch_tot_nhat+1:>3}: val loss = {min(ls_dai['val_loss']):.4f}"
      f"   val acc = {ls_dai['val_acc'][epoch_tot_nhat]:.4f}")
print(f"  Epoch cuối     {cuoi:>3}: val loss = {ls_dai['val_loss'][-1]:.4f}"
      f"   val acc = {ls_dai['val_acc'][-1]:.4f}")
print()
print(f"  Train tiếp {cuoi - epoch_tot_nhat - 1} epoch nữa chỉ làm val loss XẤU ĐI.")

> 🔍 **Chú ý một chi tiết tinh tế trong kết quả trên:** val **loss** xấu đi (0.33 → 0.43) nhưng val **accuracy** lại nhích lên (86.4% → 88.2%).
>
> Điều này xảy ra thật và rất hay gặp: model trở nên **tự tin hơn**, nên những mẫu nó đoán đúng thì đúng chắc hơn (kéo accuracy lên), còn những mẫu nó đoán sai thì sai với xác suất cao hơn (bị cross-entropy phạt rất nặng, kéo loss lên).
>
> **Vậy dừng theo cái nào?** Theo **chỉ số bạn thật sự quan tâm**:
> - Chỉ cần nhãn đúng/sai → dừng theo **accuracy**
> - Cần *xác suất* đáng tin (để chọn ngưỡng, xếp hạng, tính chi phí như Phase 04) → dừng theo **loss**
>
> Đây là một ví dụ nữa cho bài học xuyên suốt: **chọn metric theo bài toán, không theo thói quen.**

```python
tot_nhat, kien_nhan = float("inf"), 0
for epoch in range(so_epoch):
    ...
    if val_loss < tot_nhat:
        tot_nhat, kien_nhan = val_loss, 0
        torch.save(model.state_dict(), "tot_nhat.pt")     # LƯU checkpoint
    else:
        kien_nhan += 1
        if kien_nhan >= 5:
            break

model.load_state_dict(torch.load("tot_nhat.pt"))          # nạp lại bản tốt nhất
```

> 🔑 Nhờ vậy bạn luôn giữ được model **tại điểm val tốt nhất**, không phải model ở epoch cuối (thường đã overfit).

## 7. Lưu & nạp model

In [ ]:
import tempfile

model = tao_cnn(dropout=0.3)
tmp = Path(tempfile.mkdtemp()) / "model.pt"

# ✅ Cách nên dùng: lưu state_dict
torch.save(model.state_dict(), tmp)

model_moi = tao_cnn(dropout=0.3)          # phải tạo lại ĐÚNG kiến trúc
model_moi.load_state_dict(torch.load(tmp))
model_moi.eval()

print(f"  Đã lưu ({tmp.stat().st_size/1e3:.0f} KB) và nạp lại thành công")
print(f"  Kiểm tra: {tuple(model_moi(torch.randn(2,1,28,28)).shape)}")

| Cách | Ưu | Nhược |
|---|---|---|
| `torch.save(model.state_dict())` ✅ | Nhẹ, an toàn, tương thích tốt | Phải tạo lại đúng kiến trúc khi nạp |
| `torch.save(model)` | Tiện | Gắn chặt vào cấu trúc thư mục & phiên bản, dễ hỏng |

> **Luôn dùng `state_dict`.** Và nhớ `model.eval()` sau khi nạp — nếu không dropout vẫn bật khi suy luận.

**Lưu checkpoint đầy đủ để train tiếp:**
```python
torch.save({"epoch": e, "model": model.state_dict(),
            "opt": opt.state_dict(), "loss": val_loss}, "ckpt.pt")
```

## 8. GPU

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Thiết bị:", device)

model = tao_cnn().to(device)
xb = X_train[:64].to(device)              # model VÀ dữ liệu phải CÙNG thiết bị

with torch.no_grad():
    print("Chạy được:", tuple(model(xb).shape))

if device == "cpu":
    print("\n  Không có GPU — Phase 05 vẫn chạy tốt trên CPU.")
    print("  Cần GPU thì dùng Google Colab (miễn phí):")
    print("    colab.research.google.com → Runtime → Change runtime type → T4 GPU")

In [ ]:
# Lỗi kinh điển: model ở GPU, dữ liệu ở CPU (hoặc ngược lại)
print("""
    RuntimeError: Expected all tensors to be on the same device,
    but found at least two devices, cuda:0 and cpu!

Cách sửa: .to(device) cho CẢ model VÀ mọi tensor đầu vào.

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        ...
""")

**GPU nhanh hơn bao nhiêu?**

| Tình huống | Tăng tốc |
|---|---|
| CNN, batch lớn | **10–50×** |
| Transformer lớn | **20–100×** |
| Mạng nhỏ, batch nhỏ | Đôi khi **chậm hơn** (chi phí chuyển dữ liệu) |

> 💡 Phase 05 của khoá học này chạy tốt trên CPU (CNN train ~20 giây). Colab chỉ cần cho project P3 nếu bạn muốn dùng ảnh thật.

**Kiểm tra GPU trên Colab:** `!nvidia-smi`

## 9. Transfer learning — cầu nối sang thế giới LLM

Với dữ liệu thật, hiếm khi bạn train từ đầu:

```python
from torchvision import models

model = models.resnet18(weights="IMAGENET1K_V1")   # đã học từ 1.2 triệu ảnh

for p in model.parameters():
    p.requires_grad = False                        # ĐÓNG BĂNG phần đã học

model.fc = nn.Linear(model.fc.in_features, so_lop) # thay lớp cuối
# chỉ lớp mới này được train
```

**Vì sao hiệu quả:** các lớp đầu học cạnh, góc, kết cấu — thứ **dùng chung** cho mọi loại ảnh (bạn đã thấy chúng ở bài 3). Chỉ lớp cuối cần học riêng cho bài toán của bạn.

Với 1.000 ảnh, transfer learning cho kết quả tốt hơn **rất nhiều** so với train từ đầu.

**Ba mức độ:**

| Mức | Làm gì | Dùng khi |
|---|---|---|
| **Feature extraction** | Đóng băng hết, chỉ train lớp cuối | Dữ liệu rất ít (< 1.000) |
| **Fine-tune một phần** | Mở băng vài lớp cuối | Dữ liệu vừa |
| **Fine-tune toàn bộ** | Train hết với lr nhỏ | Dữ liệu nhiều, khác miền |

> 🔗 **Đây chính là ý tưởng nền tảng của toàn bộ ngành LLM.**
>
> Bạn không train GPT từ đầu (tốn hàng chục triệu đô). Bạn lấy model đã pretrain rồi:
> - **Prompt** ← Phase 07 (tương đương "dùng luôn, không train")
> - **RAG** ← Phase 08 (đưa thêm ngữ cảnh vào lúc chạy)
> - **Fine-tune / LoRA** ← Phase 10 (tương đương "mở băng vài lớp")
>
> Ba lựa chọn đó **song ánh** với ba mức transfer learning ở trên. Hiểu bảng này là hiểu trước một nửa Phase 07–10.

## 10. Tổng kết Phase 05

**Chống overfit — theo thứ tự nên thử:**
```
① Thêm dữ liệu             ← hiệu quả nhất, thường không có sẵn
② Data augmentation         ← "dữ liệu miễn phí", hiệu quả nhất với ảnh
③ Early stopping            ← rẻ nhất, luôn nên dùng
④ Dropout / Weight decay
⑤ Model đơn giản hơn
⑥ Transfer learning         ← khi dữ liệu ít
```

**Năm lỗi PyTorch phải nhớ:**
```
❌ torch.tensor([1, 2, 3])            → int64, lỗi kiểu
❌ Quên optimizer.zero_grad()         → gradient cộng dồn
❌ Quên model.eval() khi đánh giá     → dropout vẫn bật, kết quả sai
❌ Thêm Softmax + CrossEntropyLoss    → softmax hai lần
❌ Model ở GPU, dữ liệu ở CPU         → RuntimeError device
```

## 🎯 Tiếp theo

Xong bài này → làm [**P3 — Image Classifier + Gradio**](../../../projects/P3-image-classifier/README.md).